In [16]:
"""
VALIDADOR DE CALIDAD DE DATOS - TFM
Verifica integridad y coherencia de los datos sintéticos
"""

import pandas as pd
import numpy as np
from datetime import datetime
import os

print("=" * 60)
print("🔍 VALIDADOR DE CALIDAD DE DATOS")
print("=" * 60)

🔍 VALIDADOR DE CALIDAD DE DATOS


In [17]:
# Cargar archivos CSV
print("\n📂 Cargando archivos CSV...")

try:
    df_clientes = pd.read_csv('../datos/clientes.csv')
    df_facturas = pd.read_csv('../datos/facturas.csv')
    df_historial = pd.read_csv('../datos/historial_pagos.csv')
    print("✅ Archivos cargados correctamente")
    print(f"   - Clientes: {len(df_clientes)} registros")
    print(f"   - Facturas: {len(df_facturas)} registros")
    print(f"   - Historial: {len(df_historial)} registros")
except FileNotFoundError as e:
    print(f"❌ ERROR: No se encontraron los archivos CSV")
    print(f"   {e}")

# Convertir fechas a datetime
df_facturas['fecha_emision'] = pd.to_datetime(df_facturas['fecha_emision'])
df_facturas['vencimiento'] = pd.to_datetime(df_facturas['vencimiento'])

print("\n✅ Fechas convertidas a formato datetime")


📂 Cargando archivos CSV...
✅ Archivos cargados correctamente
   - Clientes: 100 registros
   - Facturas: 500 registros
   - Historial: 100 registros

✅ Fechas convertidas a formato datetime


In [18]:
print("\n" + "-" * 60)
print("✅ VALIDACIÓN 1: Valores nulos")
print("-" * 60)

errores = False

for nombre, df in [('Clientes', df_clientes), ('Facturas', df_facturas), ('Historial', df_historial)]:
    nulos = df.isnull().sum()
    if nulos.sum() > 0:
        print(f"❌ {nombre} tiene valores nulos:")
        print(nulos[nulos > 0])
        errores = True
    else:
        print(f"✅ {nombre}: Sin valores nulos")


------------------------------------------------------------
✅ VALIDACIÓN 1: Valores nulos
------------------------------------------------------------
✅ Clientes: Sin valores nulos
✅ Facturas: Sin valores nulos
✅ Historial: Sin valores nulos


In [19]:
print("\n" + "-" * 60)
print("✅ VALIDACIÓN 2: Integridad referencial")
print("-" * 60)

# Verificar que todos los cliente_id de facturas existen en clientes
clientes_validos = set(df_clientes['cliente_id'])
clientes_en_facturas = set(df_facturas['cliente_id'])
clientes_huerfanos = clientes_en_facturas - clientes_validos

if clientes_huerfanos:
    print(f"❌ Hay {len(clientes_huerfanos)} clientes en facturas sin registro en tabla clientes")
    errores = True
else:
    print(f"✅ Todos los clientes en facturas existen en tabla clientes")

# Verificar historial
clientes_en_historial = set(df_historial['cliente_id'])
if clientes_validos != clientes_en_historial:
    print(f"❌ Desajuste entre clientes y historial")
    errores = True
else:
    print(f"✅ Historial coincide con tabla clientes")


------------------------------------------------------------
✅ VALIDACIÓN 2: Integridad referencial
------------------------------------------------------------
✅ Todos los clientes en facturas existen en tabla clientes
✅ Historial coincide con tabla clientes


In [20]:
print("\n" + "-" * 60)
print("✅ VALIDACIÓN 3: Validez de importes")
print("-" * 60)

if (df_facturas['importe'] <= 0).any():
    print(f"❌ Hay {(df_facturas['importe'] <= 0).sum()} facturas con importe <= 0")
    errores = True
else:
    print(f"✅ Todos los importes son positivos")

print(f"   Rango: {df_facturas['importe'].min():.2f}€ - {df_facturas['importe'].max():.2f}€")
print(f"   Promedio: {df_facturas['importe'].mean():.2f}€")
print(f"   Mediana: {df_facturas['importe'].median():.2f}€")


------------------------------------------------------------
✅ VALIDACIÓN 3: Validez de importes
------------------------------------------------------------
✅ Todos los importes son positivos
   Rango: 230.61€ - 11789.31€
   Promedio: 3672.88€
   Mediana: 3308.97€


In [21]:
print("\n" + "-" * 60)
print("✅ VALIDACIÓN 4: Coherencia de fechas")
print("-" * 60)

# Vencimiento debe ser >= emisión
fechas_incoherentes = df_facturas[df_facturas['vencimiento'] < df_facturas['fecha_emision']]
if len(fechas_incoherentes) > 0:
    print(f"❌ Hay {len(fechas_incoherentes)} facturas con vencimiento antes de emisión")
    errores = True
else:
    print(f"✅ Todas las fechas son coherentes (vencimiento >= emisión)")

# Verificar fechas futuras
fecha_hoy = datetime.now()
facturas_futuras = df_facturas[df_facturas['fecha_emision'] > fecha_hoy]
if len(facturas_futuras) > 0:
    print(f"⚠️  Hay {len(facturas_futuras)} facturas con fecha de emisión futura")
else:
    print(f"✅ No hay facturas con fechas futuras")


------------------------------------------------------------
✅ VALIDACIÓN 4: Coherencia de fechas
------------------------------------------------------------
✅ Todas las fechas son coherentes (vencimiento >= emisión)
✅ No hay facturas con fechas futuras


In [22]:
print("\n" + "-" * 60)
print("✅ VALIDACIÓN 5: Lógica de negocio")
print("-" * 60)

# Facturas pagadas NO deben tener días de retraso
facturas_pagadas_con_retraso = df_facturas[
    (df_facturas['pagada'] == True) & (df_facturas['dias_retraso'] > 0)
]
if len(facturas_pagadas_con_retraso) > 0:
    print(f"❌ Hay {len(facturas_pagadas_con_retraso)} facturas pagadas con retraso > 0")
    errores = True
else:
    print(f"✅ Facturas pagadas tienen días_retraso = 0")

# Facturas vencidas DEBEN tener días_retraso > 0
facturas_vencidas_sin_retraso = df_facturas[
    (df_facturas['estado'] == 'Vencida') & (df_facturas['dias_retraso'] == 0)
]
if len(facturas_vencidas_sin_retraso) > 0:
    print(f"❌ Hay {len(facturas_vencidas_sin_retraso)} facturas vencidas sin retraso")
    errores = True
else:
    print(f"✅ Facturas vencidas tienen días_retraso > 0")

# Facturas pendientes NO deben tener retraso
facturas_pendientes_con_retraso = df_facturas[
    (df_facturas['estado'] == 'Pendiente') & (df_facturas['dias_retraso'] > 0)
]
if len(facturas_pendientes_con_retraso) > 0:
    print(f"❌ Hay {len(facturas_pendientes_con_retraso)} facturas pendientes con retraso")
    errores = True
else:
    print(f"✅ Facturas pendientes no tienen retraso")


------------------------------------------------------------
✅ VALIDACIÓN 5: Lógica de negocio
------------------------------------------------------------
✅ Facturas pagadas tienen días_retraso = 0
✅ Facturas vencidas tienen días_retraso > 0
✅ Facturas pendientes no tienen retraso


In [23]:
print("\n" + "-" * 60)
print("✅ VALIDACIÓN 6: Distribuciones realistas")
print("-" * 60)

# Tasa de morosidad
tasa_morosidad = (df_facturas['estado'] == 'Vencida').mean()
print(f"   Tasa de morosidad: {tasa_morosidad:.1%}")

if tasa_morosidad < 0.15:
    print(f"⚠️  Tasa muy baja ({tasa_morosidad:.1%}) - Poco realista")
elif tasa_morosidad > 0.60:
    print(f"⚠️  Tasa muy alta ({tasa_morosidad:.1%}) - Poco realista")
else:
    print(f"✅ Tasa dentro de rango esperado (15-60%)")

# Distribución scoring
print(f"\n   Distribución scoring externo:")
print(df_clientes['scoring_externo'].value_counts(normalize=True).apply(lambda x: f"{x:.1%}"))

# Distribución sectores
print(f"\n   Top 3 sectores:")
print(df_clientes['sector'].value_counts().head(3))


------------------------------------------------------------
✅ VALIDACIÓN 6: Distribuciones realistas
------------------------------------------------------------
   Tasa de morosidad: 47.4%
✅ Tasa dentro de rango esperado (15-60%)

   Distribución scoring externo:
scoring_externo
Medio    43.0%
Alto     32.0%
Bajo     25.0%
Name: proportion, dtype: object

   Top 3 sectores:
sector
Construcción    20
Transporte      18
Alimentación    16
Name: count, dtype: int64


In [24]:
print("\n" + "-" * 60)
print("✅ VALIDACIÓN 7: Estadísticas descriptivas")
print("-" * 60)

print("\n📊 Días de retraso (solo facturas vencidas):")
facturas_vencidas = df_facturas[df_facturas['estado'] == 'Vencida']
if len(facturas_vencidas) > 0:
    print(f"   Media: {facturas_vencidas['dias_retraso'].mean():.1f} días")
    print(f"   Mediana: {facturas_vencidas['dias_retraso'].median():.1f} días")
    print(f"   Máximo: {facturas_vencidas['dias_retraso'].max():.0f} días")
    print(f"   Percentil 75: {facturas_vencidas['dias_retraso'].quantile(0.75):.1f} días")

print("\n📊 Historial de pagos:")
print(f"   Tasa promedio pago a tiempo: {df_historial['tasa_pago_a_tiempo'].mean():.1%}")
print(f"   Promedio días retraso histórico: {df_historial['avg_dias_retraso'].mean():.1f} días")


------------------------------------------------------------
✅ VALIDACIÓN 7: Estadísticas descriptivas
------------------------------------------------------------

📊 Días de retraso (solo facturas vencidas):
   Media: 78.8 días
   Mediana: 81.0 días
   Máximo: 159 días
   Percentil 75: 116.0 días

📊 Historial de pagos:
   Tasa promedio pago a tiempo: 70.2%
   Promedio días retraso histórico: 15.7 días


In [25]:
print("\n" + "=" * 60)
if errores:
    print("❌ VALIDACIÓN FALLIDA - Hay errores críticos")
    print("   Revisa el generador y vuelve a ejecutar")
else:
    print("✅ VALIDACIÓN EXITOSA - Datos listos para análisis")
    print("   Puedes continuar con el EDA y modelo ML")
print("=" * 60)



✅ VALIDACIÓN EXITOSA - Datos listos para análisis
   Puedes continuar con el EDA y modelo ML
